# 1. Imports and inspect

In [49]:
from pathlib import Path
from datasets import load_dataset, get_dataset_config_names, Dataset, DatasetDict
import pandas as pd
import json
import random
import re


pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 300)
pd.set_option("display.width", 200)

SEED = 42
random.seed(SEED)

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

OUT_DIR = REPO_ROOT / "data" / "processed"
print("Repo root:", REPO_ROOT)
print("Output root:", OUT_DIR)

Repo root: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual
Output root: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed


In [38]:
# LATAMQA_DATASETS = {
#     "latamqa_es_la": "inria-chile/latamqa_mcq_es-la",
#     "latamqa_es_es": "inria-chile/latamqa_mcq_es-es",
# }
LATAMQA_DATASETS = {
    "latamqa_mcq_blind": "inria-chile/latamqa_mcq_blind",
}

BLEND_DATASET = "nayeon212/BLEnD"
N_EXAMPLES_PER_COUNTRY = 3
SEED = 42

In [37]:
def safe_get_configs(dataset_name):
    try:
        configs = get_dataset_config_names(dataset_name)
        print(f"{dataset_name}: {len(configs)} configs")
        for cfg in configs:
            print(" ", cfg)
        return configs
    except Exception as e:
        print(f"Could not list configs for {dataset_name}: {type(e).__name__}: {e}")
        return None


blend_configs = safe_get_configs(BLEND_DATASET)

nayeon212/BLEnD: 3 configs
  annotations
  short-answer-questions
  multiple-choice-questions


In [10]:
def inspect_datasetdict(ds, name):
    print("=" * 100)
    print(name)
    print(ds)

    for split_name, split_ds in ds.items():
        print("-" * 100)
        print(f"Split: {split_name}")
        print(f"Rows: {len(split_ds)}")
        print("Columns:", split_ds.column_names)

        if len(split_ds) > 0:
            print("First row:")
            print(split_ds[0])


def datasetdict_to_small_df(ds, max_rows_per_split=5):
    rows = []
    for split_name, split_ds in ds.items():
        n = min(max_rows_per_split, len(split_ds))
        for i in range(n):
            row = dict(split_ds[i])
            row["_split"] = split_name
            row["_row_idx"] = i
            rows.append(row)
    return pd.DataFrame(rows)

In [39]:
def get_dataset_from_maybe_datasetdict(obj, split="test"):
    if isinstance(obj, DatasetDict):
        return obj[split]
    if isinstance(obj, Dataset):
        return obj
    raise TypeError(f"Unexpected dataset type: {type(obj)}")

# 2. Inspect

## Blend

In [40]:
BLEND_CONFIG = "multiple-choice-questions"

try:
    blend_raw = load_dataset(BLEND_DATASET, BLEND_CONFIG)
    inspect_datasetdict(blend_raw, f"{BLEND_DATASET} / {BLEND_CONFIG}")
except Exception as e:
    print(f"Failed to load BLEnD with config={BLEND_CONFIG}: {type(e).__name__}: {e}")
    blend_raw = None

nayeon212/BLEnD / multiple-choice-questions
DatasetDict({
    test: Dataset({
        features: ['MCQID', 'ID', 'country', 'prompt', 'choices', 'choice_countries', 'answer_idx'],
        num_rows: 305939
    })
})
----------------------------------------------------------------------------------------------------
Split: test
Rows: 305939
Columns: ['MCQID', 'ID', 'country', 'prompt', 'choices', 'choice_countries', 'answer_idx']
First row:
{'MCQID': 'Al-en-01_0', 'ID': 'Al-en-01', 'country': 'UK', 'prompt': 'What is a common snack for preschool kids in the UK? Without any explanation, choose only one from the given alphabet choices(e.g., A, B, C). Provide as JSON format: {"answer_choice":""}\n\nA. candy\nB. cookie\nC. egg\nD. fruit\n\nAnswer:', 'choices': '{\n    "A": "candy",\n    "B": "cookie",\n    "C": "egg",\n    "D": "fruit"\n}', 'choice_countries': '{\n    "A": "North_Korea",\n    "B": "South_Korea",\n    "C": "Mexico",\n    "D": "UK"\n}', 'answer_idx': 'D'}


In [13]:
if blend_raw is not None:
    display(datasetdict_to_small_df(blend_raw, max_rows_per_split=10))

,MCQID,ID,country,prompt,choices,choice_countries,answer_idx,_split,_row_idx
0,Al-en-01_0,Al-en-01,UK,"What is a common snack for preschool kids in the UK? Without any explanation, choose only one from the given alphabet choices(e.g., A, B, C). Provide as JSON format: {""answer_choice"":""""}\n\nA. candy\nB. cookie\nC. egg\nD. fruit\n\nAnswer:","{\n ""A"": ""candy"",\n ""B"": ""cookie"",\n ""C"": ""egg"",\n ""D"": ""fruit""\n}","{\n ""A"": ""North_Korea"",\n ""B"": ""South_Korea"",\n ""C"": ""Mexico"",\n ""D"": ""UK""\n}",D,test,0
1,Al-en-01_1,Al-en-01,UK,"What is a common snack for preschool kids in the UK? Without any explanation, choose only one from the given alphabet choices(e.g., A, B, C). Provide as JSON format: {""answer_choice"":""""}\n\nA. cookie\nB. egg\nC. fruit\nD. jelly\n\nAnswer:","{\n ""A"": ""cookie"",\n ""B"": ""egg"",\n ""C"": ""fruit"",\n ""D"": ""jelly""\n}","{\n ""A"": ""South_Korea"",\n ""B"": ""Mexico"",\n ""C"": ""UK"",\n ""D"": ""West_Java""\n}",C,test,1
2,Al-en-01_2,Al-en-01,UK,"What is a common snack for preschool kids in the UK? Without any explanation, choose only one from the given alphabet choices(e.g., A, B, C). Provide as JSON format: {""answer_choice"":""""}\n\nA. cookie\nB. egg\nC. fruit\nD. jam sandwiches\n\nAnswer:","{\n ""A"": ""cookie"",\n ""B"": ""egg"",\n ""C"": ""fruit"",\n ""D"": ""jam sandwiches""\n}","{\n ""A"": ""South_Korea"",\n ""B"": ""Mexico"",\n ""C"": ""UK"",\n ""D"": ""Algeria""\n}",C,test,2
3,Al-en-01_3,Al-en-01,UK,"What is a common snack for preschool kids in the UK? Without any explanation, choose only one from the given alphabet choices(e.g., A, B, C). Provide as JSON format: {""answer_choice"":""""}\n\nA. chocolate paste\nB. cookie\nC. egg\nD. fruit\n\nAnswer:","{\n ""A"": ""chocolate paste"",\n ""B"": ""cookie"",\n ""C"": ""egg"",\n ""D"": ""fruit""\n}","{\n ""A"": ""Indonesia"",\n ""B"": ""South_Korea"",\n ""C"": ""Mexico"",\n ""D"": ""UK""\n}",D,test,3
4,Al-en-01_4,Al-en-01,UK,"What is a common snack for preschool kids in the UK? Without any explanation, choose only one from the given alphabet choices(e.g., A, B, C). Provide as JSON format: {""answer_choice"":""""}\n\nA. cookie\nB. egg\nC. fruit\nD. semolina\n\nAnswer:","{\n ""A"": ""cookie"",\n ""B"": ""egg"",\n ""C"": ""fruit"",\n ""D"": ""semolina""\n}","{\n ""A"": ""South_Korea"",\n ""B"": ""Mexico"",\n ""C"": ""UK"",\n ""D"": ""Assam""\n}",C,test,4
5,Al-en-01_5,Al-en-01,UK,"What is a common snack for preschool kids in the UK? Without any explanation, choose only one from the given alphabet choices(e.g., A, B, C). Provide as JSON format: {""answer_choice"":""""}\n\nA. cookie\nB. egg\nC. fruit\nD. potato fries\n\nAnswer:","{\n ""A"": ""cookie"",\n ""B"": ""egg"",\n ""C"": ""fruit"",\n ""D"": ""potato fries""\n}","{\n ""A"": ""South_Korea"",\n ""B"": ""Mexico"",\n ""C"": ""UK"",\n ""D"": ""Ethiopia""\n}",C,test,5
6,Al-en-01_6,Al-en-01,UK,"What is a common snack for preschool kids in the UK? Without any explanation, choose only one from the given alphabet choices(e.g., A, B, C). Provide as JSON format: {""answer_choice"":""""}\n\nA. cookie\nB. egg\nC. fruit\nD. noodles\n\nAnswer:","{\n ""A"": ""cookie"",\n ""B"": ""egg"",\n ""C"": ""fruit"",\n ""D"": ""noodles""\n}","{\n ""A"": ""South_Korea"",\n ""B"": ""Mexico"",\n ""C"": ""UK"",\n ""D"": ""Northern_Nigeria""\n}",C,test,6
7,Al-en-01_7,Al-en-01,UK,"What is a common snack for preschool kids in the UK? Without any explanation, choose only one from the given alphabet choices(e.g., A, B, C). Provide as JSON format: {""answer_choice"":""""}\n\nA. candy\nB. cookie\nC. fruit\nD. jelly\n\nAnswer:","{\n ""A"": ""candy"",\n ""B"": ""cookie"",\n ""C"": ""fruit"",\n ""D"": ""jelly""\n}","{\n ""A"": ""North_Korea"",\n ""B"": ""South_Korea"",\n ""C"": ""UK"",\n ""D"": ""West_Java""\n}",C,test,7
8,Al-en-01_8,Al-en-01,UK,"What is a common snack for preschool kids in the UK? Without any explanation, choose only one from the given alphabet choices(e.g., A, B, C). Provide as JSON fo

In [14]:
if blend_raw is not None:
    for split_name, split_ds in blend_raw.items():
        print("=" * 100)
        print(f"BLEnD split: {split_name}")

        df_sample = split_ds.to_pandas()
        print("Columns:", list(df_sample.columns))
        print("Rows:", len(df_sample))

        for col in df_sample.columns:
            col_lower = col.lower()
            if any(key in col_lower for key in ["country", "region", "lang", "locale", "culture"]):
                print(f"\nValue counts for column: {col}")
                print(df_sample[col].value_counts(dropna=False).head(30).to_string())

BLEnD split: test
Columns: ['MCQID', 'ID', 'country', 'prompt', 'choices', 'choice_countries', 'answer_idx']
Rows: 305939

Value counts for column: country
country
Ethiopia            22712
South_Korea         21439
Assam               21293
Mexico              20513
China               20410
Greece              20383
Algeria             20364
Azerbaijan          19932
Iran                19371
Spain               19280
Indonesia           18417
North_Korea         17005
UK                  16723
US                  16491
Northern_Nigeria    16317
West_Java           15289


In [19]:
BLEND_COUNTRIES = {
    "Mexico",
    "Spain",
    "China",
    "India",
    "Russia",
}

In [31]:
blend_ds = get_dataset_from_maybe_datasetdict(blend_raw, split="test")
blend_df = blend_ds.to_pandas()

print("BLEnD columns:", list(blend_df.columns))
print("Total BLEnD rows:", len(blend_df))

blend_selected = blend_df[blend_df["country"].isin(BLEND_COUNTRIES)].copy()

print("\nSelected BLEnD rows:", len(blend_selected))
print("\nCounts by country:")
print(blend_selected["country"].value_counts().to_string())

missing_blend = sorted(BLEND_COUNTRIES - set(blend_selected["country"].unique()))
print("\nSelected countries with 0 rows:")
print(missing_blend)

BLEnD columns: ['MCQID', 'ID', 'country', 'prompt', 'choices', 'choice_countries', 'answer_idx']
Total BLEnD rows: 305939

Selected BLEnD rows: 60203

Counts by country:
country
Mexico    20513
China     20410
Spain     19280

Selected countries with 0 rows:
['India', 'Russia']


In [32]:
# BLEnD — examples per selected country

for country in sorted(BLEND_COUNTRIES):
    sub = blend_selected[blend_selected["country"] == country].copy()

    print("=" * 100)
    print(f"BLEnD country: {country}")
    print(f"Rows: {len(sub)}")

    if len(sub) == 0:
        continue

    sample = sub.sample(n=min(N_EXAMPLES_PER_COUNTRY, len(sub)), random_state=SEED)

    for _, row in sample.iterrows():
        print("-" * 100)
        print(f"MCQID: {row.get('MCQID')}")
        print(f"ID: {row.get('ID')}")
        print(f"country: {row.get('country')}")
        print(f"answer_idx: {row.get('answer_idx')}")
        print("prompt:")
        print(row.get("prompt"))
        print("choices:")
        print(row.get("choices"))
        print("choice_countries:")
        print(row.get("choice_countries"))

BLEnD country: China
Rows: 20410
----------------------------------------------------------------------------------------------------
MCQID: New-ha-07_817
ID: New-ha-07
country: China
answer_idx: A
prompt:
What food originated from China and can be found all over the world? Without any explanation, choose only one from the given alphabet choices(e.g., A, B, C). Provide as JSON format: {"answer_choice":""}

A. dumplings
B. fish & chips
C. fried rice
D. tacos

Answer:
choices:
{
    "A": "dumplings",
    "B": "fish & chips",
    "C": "fried rice",
    "D": "tacos"
}
choice_countries:
{
    "A": "China",
    "B": "UK",
    "C": "Indonesia",
    "D": "Mexico"
}
----------------------------------------------------------------------------------------------------
MCQID: New-ch-04_128
ID: New-ch-04
country: China
answer_idx: B
prompt:
What kinds of food do people usually eat for late-night snacks in China? Without any explanation, choose only one from the given alphabet choices(e.g., A, B, C).

# 3. Process

In [78]:
import ast
import json
import random
import re
from collections import Counter
import pandas as pd

BLEND_GROUP_COUNTRIES = {
    "english_spain_mexico": {"Mexico", "Spain"},
    "english_china": {"China"},
}

BLEND_COUNTRIES = set().union(*BLEND_GROUP_COUNTRIES.values())

BLEND_OUT_DIR = OUT_DIR / "blend"
BLEND_OUT_DIR.mkdir(parents=True, exist_ok=True)

N_EXAMPLES = 20

In [71]:
# helpers
PROMPT_INSTRUCTION_RE = re.compile(
    r"\s*Without any explanation, choose only one from the given alphabet choices"
    r"\(e\.g\., A, B, C\)\. Provide as JSON format:\s*\{[\"']answer_choice[\"']:\s*[\"'][\"']\}",
    flags=re.IGNORECASE,
)

TRAILING_ANSWER_RE = re.compile(r"\s*Answer:\s*$", flags=re.IGNORECASE)

INLINE_OPTIONS_RE = re.compile(
    r"\s+A\.\s+.*?\s+B\.\s+.*?\s+C\.\s+.*?\s+D\.\s+.*?(?=\s*Answer:\s*$|$)",
    flags=re.IGNORECASE | re.DOTALL,
)

BAD_ARTIFACT_RE = re.compile(
    r"json|csv|answer_choice|without any explanation|provide as json|answer:\s*$|\bA\.|\bB\.|\bC\.|\bD\.",
    flags=re.IGNORECASE,
)


def parse_choices(x):
    if isinstance(x, dict):
        return {str(k).strip().upper(): str(v).strip() for k, v in x.items()}
    if isinstance(x, str):
        s = x.strip()
        try:
            parsed = json.loads(s)
        except json.JSONDecodeError:
            parsed = ast.literal_eval(s)
        if not isinstance(parsed, dict):
            raise ValueError(f"choices is not a dict: {type(parsed)}")
        return {str(k).strip().upper(): str(v).strip() for k, v in parsed.items()}
    raise ValueError(f"unexpected choices type: {type(x)}")


def clean_blend_prompt(prompt):
    text = str(prompt).strip()
    text = PROMPT_INSTRUCTION_RE.sub("", text)
    text = INLINE_OPTIONS_RE.sub("", text)
    text = TRAILING_ANSWER_RE.sub("", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_text(x):
    return re.sub(r"\s+", " ", str(x).strip()).lower()

In [79]:
blend_df = blend_raw["test"].to_pandas()
blend_df = blend_df[blend_df["country"].isin(BLEND_COUNTRIES)].copy()
blend_df = blend_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("selected rows:", len(blend_df))
print(blend_df["country"].value_counts().to_string())

selected rows: 60203
country
Mexico    20513
China     20410
Spain     19280


In [80]:
rows = []
stats = Counter()

for _, row in blend_df.iterrows():
    try:
        choices = parse_choices(row["choices"])
    except Exception:
        stats["removed_bad_choices_parse"] += 1
        continue

    prompt_clean = clean_blend_prompt(row["prompt"])
    answer = str(row["answer_idx"]).strip().upper()

    if not prompt_clean:
        stats["removed_empty_prompt"] += 1
        continue
    if BAD_ARTIFACT_RE.search(prompt_clean):
        stats["removed_prompt_artifact"] += 1
        continue
    if answer not in choices:
        stats["removed_answer_not_in_choices"] += 1
        continue
    if len(choices) < 2:
        stats["removed_too_few_choices"] += 1
        continue
    if len(set(normalize_text(v) for v in choices.values())) != len(choices):
        stats["removed_duplicate_choice_texts"] += 1
        continue

    out = {
        "MCQID": row.get("MCQID", ""),
        "ID": row.get("ID", ""),
        "country": row.get("country", ""),
        "prompt_clean": prompt_clean,
        "answer_idx": answer,
        "n_choices": len(choices),
    }

    for letter in ["A", "B", "C", "D", "E"]:
        out[f"option_{letter}"] = choices.get(letter, "")

    rows.append(out)

blend_flat = pd.DataFrame(rows)

print("kept after cleaning:", len(blend_flat))
print("removed:")
for key, value in sorted(stats.items()):
    print(f"  {key}: {value}")

display(blend_flat.head())

kept after cleaning: 60203
removed:


,MCQID,ID,country,prompt_clean,answer_idx,n_choices,option_A,option_B,option_C,option_D,option_E
0,Gu-ch-06_199,Gu-ch-06,Mexico,What sports do men like to play the most in Mexico?,D,4,American Football,badminton,cricket,soccer,
1,New-ch-27_349,New-ch-27,China,Who is the most famous boxer in China?,D,4,"""canelo"" alvarez",chris john,hong soo-hwan,zou shiming,
2,Sa-en-16_2248,Sa-en-16,Mexico,What is the most popular soccer team among the people from Mexico?,A,4,america,guangzhou fc,mca,not-applicable,
3,New-en-69_2067,New-en-69,Spain,Which writers are commonly studied in literature class in Spain?,A,4,cervantes,gabriel garcía márquez,haddis alemayehu,william shakespeare,
4,Th-en-24_2491,Th-en-24,Spain,What region in Spain is a major hub for tech workers and start ups?,A,4,barcelona,jakarta,kano,london,


In [81]:
dedup_cols = [
    "prompt_clean",
    "option_A",
    "option_B",
    "option_C",
    "option_D",
    "option_E",
    "answer_idx",
]

before = len(blend_flat)
blend_flat = blend_flat.drop_duplicates(subset=dedup_cols).reset_index(drop=True)

print(f"deduplicated: {before} -> {len(blend_flat)}")
print(blend_flat["country"].value_counts().to_string())

deduplicated: 60203 -> 60203
country
Mexico    20513
China     20410
Spain     19280


In [82]:
def make_blend_prompt(row):
    lines = [str(row["prompt_clean"]).strip(), ""]

    for letter in ["A", "B", "C", "D", "E"]:
        option = str(row.get(f"option_{letter}", "")).strip()
        if option:
            lines.append(f"{letter}) {option}")

    return "\n".join(lines)


def format_blend_example(row, idx, group_name):
    answer = str(row["answer_idx"]).strip().upper()

    return {
        "id": f"blend_{group_name}_{idx}",
        "dataset": "blend",
        "language": "english",
        "task_type": "mcq",
        "prompt": make_blend_prompt(row),
        "answer": answer,
        "metadata": {
            "source_id": str(row.get("MCQID", "")),
            "group_id": str(row.get("ID", "")),
            "country": str(row.get("country", "")),
            "region_group": group_name.replace("english_", ""),
            "source_split": "test",
            "n_choices": int(row.get("n_choices", 0)),
        },
    }


examples_by_group = {}

for group_name, countries in BLEND_GROUP_COUNTRIES.items():
    group_df = blend_flat[blend_flat["country"].isin(countries)].copy().reset_index(drop=True)

    examples = [
        format_blend_example(row, idx, group_name)
        for idx, row in group_df.iterrows()
    ]

    examples_by_group[group_name] = examples
    print(f"{group_name}: {len(examples)} examples")

english_spain_mexico: 39793 examples
english_china: 20410 examples


In [84]:
rng = random.Random(SEED)
for group_name, examples in examples_by_group.items():
    print("=" * 100)
    print(f"group: {group_name}")
    print(f"total examples: {len(examples)}")

    sampled_examples = rng.sample(examples, k=min(N_EXAMPLES, len(examples)))

    for ex in sampled_examples:
        print("-" * 100)
        print(f"id: {ex['id']}")
        print(f"language: {ex['language']}")
        print(f"answer: {ex['answer']}")
        print(f"metadata: {ex['metadata']}")
        print(ex["prompt"])

group: english_spain_mexico
total examples: 39793
----------------------------------------------------------------------------------------------------
id: blend_english_spain_mexico_7296
language: english
answer: D
metadata: {'source_id': 'Na-ko-18_179', 'group_id': 'Na-ko-18', 'country': 'Mexico', 'region_group': 'spain_mexico', 'source_split': 'test', 'n_choices': 4}
What is the most common sport girls participate in during physical education classes in Mexico?

A) dance
B) dodgeball
C) jump rope
D) soccer
----------------------------------------------------------------------------------------------------
id: blend_english_spain_mexico_1639
language: english
answer: D
metadata: {'source_id': 'Jod-ch-01_173', 'group_id': 'Jod-ch-01', 'country': 'Mexico', 'region_group': 'spain_mexico', 'source_split': 'test', 'n_choices': 4}
What eating utensils do people commonly used in Mexico?

A) big plate
B) bronze utensils
C) gimchi
D) spoon
------------------------------------------------------

In [75]:
rng = random.Random(SEED)
for language, examples in examples_by_language.items():
    print("=" * 100)
    print(f"language: {language}")
    print(f"total examples: {len(examples)}")

    sampled_examples = rng.sample(examples, k=min(N_EXAMPLES, len(examples)))

    for ex in sampled_examples:
        print("-" * 100)
        print(f"id: {ex['id']}")
        print(f"answer: {ex['answer']}")
        print(f"metadata: {ex['metadata']}")
        print(ex["prompt"])

language: spanish
total examples: 39793
----------------------------------------------------------------------------------------------------
id: blend_spanish_7296
answer: C
metadata: {'source_id': 'New-ha-07_1452', 'group_id': 'New-ha-07', 'country': 'Spain', 'source_split': 'test', 'n_choices': 4}
What food originated from Spain and can be found all over the world?

A) couscous
B) fried rice
C) paella
D) tea
----------------------------------------------------------------------------------------------------
id: blend_spanish_1639
answer: B
metadata: {'source_id': 'Ta-pe-32_725', 'group_id': 'Ta-pe-32', 'country': 'Spain', 'source_split': 'test', 'n_choices': 4}
What day of the year is usually dedicated to fireworks in Spain?

A) easter
B) end of year
C) mawlid
D) spring festival's eve
----------------------------------------------------------------------------------------------------
id: blend_spanish_18024
answer: D
metadata: {'source_id': 'New-ch-38_5004', 'group_id': 'New-ch-38', 

In [ ]:
for group_name, examples in examples_by_group.items():
    jsonl_path = BLEND_OUT_DIR / f"blend_{group_name}_all.jsonl"
    with jsonl_path.open("w", encoding="utf-8") as f:
        for ex in examples:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")
    print(f"saved {len(examples)} examples to {jsonl_path}")

saved 39793 examples to /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/blend/blend_english_spain_mexico_all.jsonl
saved 20410 examples to /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/blend/blend_english_china_all.jsonl
